In [1]:

# ============================================================
# Cell 1: Install vLLM in the active Colab environment
# ============================================================
# Remove the incompatible TorchAudio package.
%pip uninstall -y torchaudio
import sys
import subprocess

# Install uv into the notebook's Python environment.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-U", "uv"],
    check=True,
)

# Install vLLM into the exact Python environment
# used by the current Jupyter kernel.
subprocess.run(
    [
        sys.executable, "-m", "uv", "pip", "install",
        "--python", sys.executable,
        "vllm",
        "--torch-backend", "auto",
    ],
    check=True,
)

print("vLLM installation completed.")


Found existing installation: torchaudio 2.11.0+cu130
Uninstalling torchaudio-2.11.0+cu130:
  Successfully uninstalled torchaudio-2.11.0+cu130
vLLM installation completed.


In [2]:

# ============================================================
# Verify vLLM installation
# ============================================================

import torch
import vllm

print("vLLM version:", vllm.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


vLLM version: 0.33.0
PyTorch version: 2.14.0+cu132
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


In [3]:

# ============================================================
# Cell 2: Verify inference environment
# ============================================================

import gc
import json
import os
import random
import re
import signal
import subprocess
import sys
import threading
import time

from pathlib import Path
from urllib.request import urlopen
from urllib.error import URLError

import httpx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import vllm

from tqdm.auto import tqdm

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

assert torch.cuda.is_available(), (
    "No CUDA GPU detected. Enable an A100 GPU in Colab."
)

GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = (
    torch.cuda.get_device_properties(0).total_memory / 1024**3
)

print("GPU:", GPU_NAME)
print("GPU memory:", round(GPU_MEMORY_GB, 2), "GiB")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("vLLM:", vllm.__version__)
print("Python:", sys.version.split()[0])


GPU: NVIDIA A100-SXM4-80GB
GPU memory: 79.25 GiB
PyTorch: 2.14.0+cu132
CUDA: 13.2
vLLM: 0.33.0
Python: 3.13.16


In [4]:

# ============================================================
# Cell 3: Locate model and evaluation artifacts
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/clinical-qa-finetuning-v2"
)

MODEL_DIR = PROJECT_DIR / "models"

RESULTS_DIR = (
    PROJECT_DIR / "results" / "vllm_benchmarks"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

GPTQ_MODEL_DIR = (
    MODEL_DIR / "qwen3-clinical-gptq-int4"
)

# Candidate locations for the merged BF16 reference.
BF16_CANDIDATES = [
    MODEL_DIR / "qwen3-clinical-merged-bf16",
    Path(
        "/content/clinical_qa_gptq/"
        "qwen3-clinical-merged-bf16"
    ),
]

BF16_MODEL_DIR = next(
    (
        path
        for path in BF16_CANDIDATES
        if (path / "config.json").is_file()
        and list(path.glob("*.safetensors"))
    ),
    None,
)


def check_model_artifacts(directory):
    """
    Verify essential files for a standalone model.

    Parameters
    ----------
    directory : Path
        Local or Drive model directory.

    Returns
    -------
    bool
        True if configuration and weight files exist.
    """
    return (
        (directory / "config.json").is_file()
        and bool(list(directory.glob("*.safetensors")))
    )


assert check_model_artifacts(GPTQ_MODEL_DIR), (
    f"GPTQ checkpoint missing: {GPTQ_MODEL_DIR}"
)

MODEL_PATHS = {
    "gptq_int4": GPTQ_MODEL_DIR,
}

if BF16_MODEL_DIR is not None:
    MODEL_PATHS["merged_bf16"] = BF16_MODEL_DIR

print("GPTQ model:", GPTQ_MODEL_DIR)
print("Merged BF16:", BF16_MODEL_DIR)
print("Available benchmark models:", list(MODEL_PATHS))
print("Results directory:", RESULTS_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPTQ model: /content/drive/MyDrive/clinical-qa-finetuning-v2/models/qwen3-clinical-gptq-int4
Merged BF16: None
Available benchmark models: ['gptq_int4']
Results directory: /content/drive/MyDrive/clinical-qa-finetuning-v2/results/vllm_benchmarks


In [5]:

# ============================================================
# Cell 4: Stage model checkpoints on local storage
# ============================================================

import shutil


def stage_model(source_dir, destination_dir):
    """
    Copy a model checkpoint to local Colab storage.

    Parameters
    ----------
    source_dir : Path
        Existing model directory.
    destination_dir : Path
        Target local directory.

    Returns
    -------
    Path
        Verified local checkpoint directory.
    """
    if source_dir.resolve() == destination_dir.resolve():
        return destination_dir

    if not destination_dir.exists():
        shutil.copytree(source_dir, destination_dir)

    source_inventory = {
        str(path.relative_to(source_dir)): path.stat().st_size
        for path in source_dir.rglob("*")
        if path.is_file()
    }

    target_inventory = {
        str(path.relative_to(destination_dir)): path.stat().st_size
        for path in destination_dir.rglob("*")
        if path.is_file()
    }

    if source_inventory != target_inventory:
        raise RuntimeError(
            f"Model staging verification failed: {destination_dir}"
        )

    return destination_dir


LOCAL_MODEL_ROOT = Path("/content/vllm_model_cache")
LOCAL_MODEL_ROOT.mkdir(parents=True, exist_ok=True)

STAGED_MODELS = {}

for model_name, source_path in MODEL_PATHS.items():
    local_path = LOCAL_MODEL_ROOT / model_name

    STAGED_MODELS[model_name] = stage_model(
        source_path,
        local_path,
    )

    print(f"{model_name}: {local_path}")

print("All available checkpoints staged successfully.")


gptq_int4: /content/vllm_model_cache/gptq_int4
All available checkpoints staged successfully.


In [6]:

# ============================================================
# Cell 5: Inspect model checkpoint metadata
# ============================================================

def model_size_gib(directory):
    """
    Calculate total checkpoint directory size.

    Parameters
    ----------
    directory : Path
        Model checkpoint directory.

    Returns
    -------
    float
        Total size in GiB.
    """
    total_bytes = sum(
        path.stat().st_size
        for path in directory.rglob("*")
        if path.is_file()
    )

    return total_bytes / 1024**3


checkpoint_sizes = {}

for model_name, model_path in STAGED_MODELS.items():
    config_path = model_path / "config.json"

    with open(config_path, encoding="utf-8") as file:
        config = json.load(file)

    size_gib = model_size_gib(model_path)

    checkpoint_sizes[model_name] = size_gib

    print("\nModel:", model_name)
    print("Size:", round(size_gib, 3), "GiB")
    print("Architecture:", config.get("architectures"))
    print(
        "Quantization configuration:",
        config.get("quantization_config", "None"),
    )



Model: gptq_int4
Size: 2.497 GiB
Architecture: ['Qwen3ForCausalLM']
Quantization configuration: {'bits': 4, 'checkpoint_format': 'gptq', 'desc_act': False, 'format': 'gptq', 'group_size': 128, 'lm_head': False, 'meta': {'act_group_aware': True, 'auto_forward_data_parallel': True, 'damp_auto_increment': 0.01, 'damp_percent': 0.05, 'dense_vram_strategy': 'exclusive', 'dense_vram_strategy_devices': None, 'fallback': {'smooth': None, 'strategy': 'rtn', 'threshold': '0.5%'}, 'foem': None, 'gc_mode': 'interval', 'gptaq': None, 'hessian': {'chunk_bytes': None, 'chunk_size': None, 'dedup_shared_inputs': True, 'staging_dtype': 'float32'}, 'mock_quantization': False, 'moe_vram_strategy': 'exclusive', 'moe_vram_strategy_devices': None, 'mse': 0.0, 'offload_to_disk': True, 'offload_to_disk_path': '/tmp/gptqmodel_bibznmx_', 'pack_impl': 'auto', 'quantizer': ['gptqmodel:7.10.0'], 'static_groups': False, 'telemetry': {'device': False}, 'true_sequential': True, 'uri': 'https://github.com/modelcloud/g

In [7]:

# ============================================================
# Cell 6: Restore clinical QA evaluation questions
# ============================================================

from transformers import AutoTokenizer

DATA_DIR = PROJECT_DIR / "data"

EVALUATION_DIR = (
    PROJECT_DIR / "results" / "evaluation"
)

SELECTION_FILE = (
    EVALUATION_DIR / "evaluation_selection.json"
)

MEDQA_FILE = DATA_DIR / "medqa_validation.jsonl"

PUBMEDQA_FILE = DATA_DIR / "pubmedqa_validation.jsonl"

for file_path in [
    SELECTION_FILE,
    MEDQA_FILE,
    PUBMEDQA_FILE,
]:
    assert file_path.is_file(), (
        f"Required evaluation file missing: {file_path}"
    )


def load_jsonl(path):
    """
    Load JSONL records without changing their contents.

    Parameters
    ----------
    path : Path
        JSONL file location.

    Returns
    -------
    list[dict]
        Parsed dataset records.
    """
    records = []

    with open(path, "r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                records.append(json.loads(line))

    return records


with open(SELECTION_FILE, "r") as file:
    selection = json.load(file)

medqa_records = load_jsonl(MEDQA_FILE)
pubmedqa_records = load_jsonl(PUBMEDQA_FILE)

medqa_index = {
    str(row["id"]): row
    for row in medqa_records
}

pubmedqa_index = {
    str(row["id"]): row
    for row in pubmedqa_records
}

medqa_selected = [
    medqa_index[str(question_id)]
    for question_id in selection["medqa_ids"]
]

pubmedqa_selected = [
    pubmedqa_index[str(question_id)]
    for question_id in selection["pubmedqa_ids"]
]

tokenizer = AutoTokenizer.from_pretrained(
    str(STAGED_MODELS["gptq_int4"])
)

print("MedQA questions:", len(medqa_selected))
print("PubMedQA questions:", len(pubmedqa_selected))


MedQA questions: 100
PubMedQA questions: 100


In [8]:

# ============================================================
# Cell 7: Create consistent model inputs
# ============================================================

MEDQA_SYSTEM = (
    "You are answering educational medical "
    "multiple-choice questions. Select the "
    "single best answer. Begin with A, B, C, or D."
)

PUBMEDQA_SYSTEM = (
    "You answer biomedical research questions "
    "using the provided abstract evidence. "
    "Begin your answer with yes, no, or maybe."
)


def make_prompt(example):
    """
    Format a clinical QA question for Qwen3 inference.

    Parameters
    ----------
    example : dict
        Normalized MedQA or PubMedQA example.

    Returns
    -------
    str
        Qwen3 non-thinking chat-formatted prompt.
    """
    task = example["task"]

    if task == "medqa":
        instruction = MEDQA_SYSTEM

    elif task == "pubmedqa":
        instruction = PUBMEDQA_SYSTEM

    else:
        raise ValueError(f"Unsupported task: {task}")

    messages = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": example["prompt"]},
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )


MAX_NEW_TOKENS = 24
MAX_MODEL_LEN = 2048

# Three separate workloads, one per concurrency level.
CONCURRENCY_LEVELS = [1, 4, 8]

QUESTIONS_PER_LEVEL = 16

assert len(medqa_selected) >= 24
assert len(pubmedqa_selected) >= 24

# Interleave the two tasks for a balanced workload.
benchmark_examples = []

for medqa_row, pubmedqa_row in zip(
    medqa_selected[:24],
    pubmedqa_selected[:24],
):
    benchmark_examples.extend([medqa_row, pubmedqa_row])

assert len(benchmark_examples) == 48

benchmark_workloads = {}

for index, concurrency in enumerate(CONCURRENCY_LEVELS):
    start = index * QUESTIONS_PER_LEVEL
    end = start + QUESTIONS_PER_LEVEL

    subset = benchmark_examples[start:end]

    benchmark_workloads[concurrency] = [
        {
            "id": str(row["id"]),
            "task": row["task"],
            "prompt": make_prompt(row),
        }
        for row in subset
    ]

    assert all(
        len(tokenizer.encode(
            item["prompt"],
            add_special_tokens=False,
        )) + MAX_NEW_TOKENS <= MAX_MODEL_LEN
        for item in benchmark_workloads[concurrency]
    )

print("Benchmark workloads prepared:")

for concurrency, workload in benchmark_workloads.items():
    print(
        f"Concurrency {concurrency}: "
        f"{len(workload)} unique requests"
    )

# Save an exact workload manifest.
manifest = {
    "seed": SEED,
    "concurrency_levels": CONCURRENCY_LEVELS,
    "questions_per_level": QUESTIONS_PER_LEVEL,
    "max_new_tokens": MAX_NEW_TOKENS,
    "max_model_len": MAX_MODEL_LEN,
    "workloads": {
        str(level): [
            {"id": row["id"], "task": row["task"]}
            for row in workload
        ]
        for level, workload in benchmark_workloads.items()
    },
}

with open(
    RESULTS_DIR / "benchmark_workload.json",
    "w",
) as file:
    json.dump(manifest, file, indent=2)


Benchmark workloads prepared:
Concurrency 1: 16 unique requests
Concurrency 4: 16 unique requests
Concurrency 8: 16 unique requests


In [9]:

# ============================================================
# Cell 8: Configure vLLM serving parameters
# ============================================================

SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8000

BASE_URL = (
    f"http://{SERVER_HOST}:{SERVER_PORT}/v1"
)

HEALTH_URL = (
    f"http://{SERVER_HOST}:{SERVER_PORT}/health"
)

SERVED_MODEL_NAME = "clinical-qa"

GPU_MEMORY_UTILIZATION = 0.80
MAX_NUM_SEQS = 16

ENABLE_PREFIX_CACHING = True

print("Server URL:", BASE_URL)
print("GPU memory utilization target:", GPU_MEMORY_UTILIZATION)
print("Maximum concurrent sequences:", MAX_NUM_SEQS)
print("Prefix caching:", ENABLE_PREFIX_CACHING)


Server URL: http://127.0.0.1:8000/v1
GPU memory utilization target: 0.8
Maximum concurrent sequences: 16
Prefix caching: True


In [10]:

# ============================================================
# Cell 9: vLLM server lifecycle management
# ============================================================

server_process = None
server_log_file = None
active_model_name = None


def server_is_healthy():
    """
    Check whether the local vLLM API server is ready.

    Returns
    -------
    bool
        True when the health endpoint responds successfully.
    """
    try:
        with urlopen(HEALTH_URL, timeout=2) as response:
            return response.status == 200

    except (URLError, TimeoutError, OSError):
        return False


def stop_vllm_server():
    """
    Stop the currently running vLLM server process.

    Returns
    -------
    None
        Terminates the process group and releases resources.
    """
    global server_process
    global server_log_file
    global active_model_name

    if server_process is not None:
        if server_process.poll() is None:
            os.killpg(
                server_process.pid,
                signal.SIGTERM,
            )

            try:
                server_process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                os.killpg(
                    server_process.pid,
                    signal.SIGKILL,
                )
                server_process.wait(timeout=10)

        server_process = None

    if server_log_file is not None:
        server_log_file.close()
        server_log_file = None

    active_model_name = None

    gc.collect()
    torch.cuda.empty_cache()


def start_vllm_server(model_name, timeout_seconds=300):
    """
    Start a vLLM OpenAI-compatible server for one checkpoint.

    Parameters
    ----------
    model_name : str
        Key in STAGED_MODELS.
    timeout_seconds : int
        Maximum server startup time.

    Returns
    -------
    None
        Waits until the health endpoint becomes available.

    Raises
    ------
    RuntimeError
        If the process exits or startup exceeds the timeout.
    """
    global server_process
    global server_log_file
    global active_model_name

    if model_name not in STAGED_MODELS:
        raise ValueError(
            f"Unavailable checkpoint: {model_name}"
        )

    stop_vllm_server()

    model_path = STAGED_MODELS[model_name]

    log_path = (
        RESULTS_DIR / f"vllm_{model_name}.log"
    )

    server_log_file = open(
        log_path,
        "w",
        encoding="utf-8",
    )

    command = [
        "vllm",
        "serve",
        str(model_path),
        "--host", SERVER_HOST,
        "--port", str(SERVER_PORT),
        "--served-model-name", SERVED_MODEL_NAME,
        "--dtype", "bfloat16",
        "--gpu-memory-utilization",
        str(GPU_MEMORY_UTILIZATION),
        "--max-model-len", str(MAX_MODEL_LEN),
        "--max-num-seqs", str(MAX_NUM_SEQS),
    ]

    if ENABLE_PREFIX_CACHING:
        command.append("--enable-prefix-caching")
    else:
        command.append("--disable-prefix-caching")

    print("Starting model:", model_name)

    server_process = subprocess.Popen(
        command,
        stdout=server_log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

    deadline = time.monotonic() + timeout_seconds

    while time.monotonic() < deadline:
        if server_process.poll() is not None:
            server_log_file.flush()

            log_tail = log_path.read_text(
                encoding="utf-8",
                errors="replace",
            )[-6000:]

            stop_vllm_server()

            raise RuntimeError(
                "vLLM server exited during startup.\n"
                + log_tail
            )

        if server_is_healthy():
            active_model_name = model_name
            print("vLLM server is ready.")
            return

        time.sleep(2)

    stop_vllm_server()

    raise RuntimeError(
        f"vLLM startup timed out. Inspect {log_path}"
    )


In [12]:
# Remove the incompatible TorchAudio package.
%pip uninstall -y torchaudio

Found existing installation: torchaudio 2.11.0+cu130
Uninstalling torchaudio-2.11.0+cu130:
  Successfully uninstalled torchaudio-2.11.0+cu130


In [24]:

# ============================================================
# Cell 10: Start GPTQ INT4 inference server
# ============================================================

start_vllm_server("gptq_int4")

assert server_is_healthy()

print("Active model:", active_model_name)
print("Endpoint:", BASE_URL)


Starting model: gptq_int4
vLLM server is ready.
Active model: gptq_int4
Endpoint: http://127.0.0.1:8000/v1


In [14]:

# ============================================================
# Cell 11: Streaming request with TTFT measurement
# ============================================================

async def stream_completion(client, item):
    """
    Execute one streamed inference request.

    Parameters
    ----------
    client : httpx.AsyncClient
        Reusable asynchronous HTTP client.
    item : dict
        Benchmark item containing id, task and prompt.

    Returns
    -------
    dict
        Request-level TTFT, latency, output tokens and text.

    Notes
    -----
    TTFT measures client-observed time to the first
    non-empty generated text chunk. It includes HTTP
    and scheduling overhead, not only model computation.
    """
    payload = {
        "model": SERVED_MODEL_NAME,
        "prompt": item["prompt"],
        "temperature": 0,
        "max_tokens": MAX_NEW_TOKENS,
        "stream": True,
        "stream_options": {
            "include_usage": True,
        },
    }

    start = time.perf_counter()

    first_token_seconds = None
    text_parts = []
    server_output_tokens = None

    async with client.stream(
        "POST",
        f"{BASE_URL}/completions",
        json=payload,
    ) as response:

        response.raise_for_status()

        async for line in response.aiter_lines():
            if not line.startswith("data: "):
                continue

            data = line[6:].strip()

            if data == "[DONE]":
                break

            chunk = json.loads(data)

            # Server-reported token usage, when available.
            usage = chunk.get("usage")

            if usage is not None:
                server_output_tokens = usage.get(
                    "completion_tokens"
                )

            for choice in chunk.get("choices", []):
                text = choice.get("text") or ""

                if text:
                    if first_token_seconds is None:
                        first_token_seconds = (
                            time.perf_counter() - start
                        )

                    text_parts.append(text)

    end_to_end_seconds = (
        time.perf_counter() - start
    )

    generated_text = "".join(text_parts)

    # Fall back to tokenizer counting only if the
    # API response does not contain completion usage.
    if server_output_tokens is None:
        output_tokens = len(
            tokenizer.encode(
                generated_text,
                add_special_tokens=False,
            )
        )
        token_count_source = "tokenizer_estimate"
    else:
        output_tokens = int(server_output_tokens)
        token_count_source = "server_usage"

    # If no text was streamed, TTFT is not measurable.
    # Do not assign an invented first-token timestamp.
    decode_seconds = None
    decode_tokens_per_second = None

    if first_token_seconds is not None:
        decode_seconds = max(
            0.0,
            end_to_end_seconds - first_token_seconds,
        )

        if decode_seconds > 0 and output_tokens > 1:
            decode_tokens_per_second = (
                (output_tokens - 1) / decode_seconds
            )

    return {
        "id": item["id"],
        "task": item["task"],
        "ttft_seconds": first_token_seconds,
        "latency_seconds": end_to_end_seconds,
        "output_tokens": output_tokens,
        "token_count_source": token_count_source,
        "decode_tokens_per_second": (
            decode_tokens_per_second
        ),
        "raw_response": generated_text,
        "success": True,
    }


In [15]:

# ============================================================
# Cell 12: Single-request inference smoke test
# ============================================================

async with httpx.AsyncClient(
    timeout=httpx.Timeout(120.0)
) as client:

    test_item = benchmark_workloads[1][0]

    result = await stream_completion(
        client,
        test_item,
    )

print("Model:", active_model_name)
print("Task:", result["task"])
print("Response:", repr(result["raw_response"]))
print("TTFT:", result["ttft_seconds"], "seconds")
print("Total latency:", result["latency_seconds"], "seconds")
print("Output tokens:", result["output_tokens"])
print("Token count:", result["token_count_source"])

assert result["success"]
assert result["latency_seconds"] > 0


Model: gptq_int4
Task: medqa
Response: 'D'
TTFT: 0.1559896490000483 seconds
Total latency: 0.15966265899987775 seconds
Output tokens: 2
Token count: server_usage


In [ ]:

# ============================================================
# Cell 13: GPU memory monitoring
# ============================================================

class GPUMemoryMonitor:
    """
    Sample total NVIDIA GPU memory usage during a workload.

    Attributes
    ----------
    samples : list[float]
        Memory measurements in MiB.
    """

    def __init__(self, interval_seconds=0.5):
        self.interval_seconds = interval_seconds
        self.samples = []
        self._stop = threading.Event()
        self._thread = None

    def _sample(self):
        """Collect GPU memory samples until monitoring stops."""
        while not self._stop.is_set():
            try:
                result = subprocess.run(
                    [
                        "nvidia-smi",
                        "--query-gpu=memory.used",
                        "--format=csv,noheader,nounits",
                    ],
                    capture_output=True,
                    text=True,
                    check=True,
                    timeout=5,
                )

                # Single-GPU Colab runtime.
                value_mib = float(
                    result.stdout.strip().splitlines()[0]
                )

                self.samples.append(value_mib)

            except (OSError, ValueError, subprocess.SubprocessError):
                pass

            self._stop.wait(self.interval_seconds)

    def start(self):
        """Begin memory sampling in a daemon thread."""
        self._stop.clear()

        self._thread = threading.Thread(
            target=self._sample,
            daemon=True,
        )

        self._thread.start()

    def stop(self):
        """
        Stop memory sampling.

        Returns
        -------
        dict
            Peak and average GPU memory in GiB.
        """
        self._stop.set()

        if self._thread is not None:
            self._thread.join(timeout=10)

        if not self.samples:
            return {
                "gpu_peak_gib": None,
                "gpu_average_gib": None,
            }

        values = np.asarray(self.samples) / 1024

        return {
            "gpu_peak_gib": float(values.max()),
            "gpu_average_gib": float(values.mean()),
        }


print("GPU memory monitor ready.")


In [16]:

# ============================================================
# Cell 14: Asynchronous concurrency benchmark
# ============================================================

async def run_concurrency_benchmark(
    model_name,
    concurrency,
    workload,
):
    """
    Benchmark one model under a fixed concurrency level.

    Parameters
    ----------
    model_name : str
        Model identifier for reporting.
    concurrency : int
        Maximum simultaneous HTTP requests.
    workload : list[dict]
        Fixed clinical QA benchmark questions.

    Returns
    -------
    tuple[dict, pandas.DataFrame]
        Summary metrics and individual request records.
    """
    assert server_is_healthy(), (
        "The vLLM server is not healthy."
    )

    semaphore = __import__("asyncio").Semaphore(
        concurrency
    )

    async def limited_request(client, item):
        """Submit one request while respecting concurrency."""
        async with semaphore:
            try:
                return await stream_completion(
                    client,
                    item,
                )

            except Exception as exc:
                return {
                    "id": item["id"],
                    "task": item["task"],
                    "ttft_seconds": None,
                    "latency_seconds": None,
                    "output_tokens": 0,
                    "token_count_source": None,
                    "decode_tokens_per_second": None,
                    "raw_response": "",
                    "success": False,
                    "error": str(exc),
                }

    monitor = GPUMemoryMonitor()
    monitor.start()

    start = time.perf_counter()

    try:
        async with httpx.AsyncClient(
            timeout=httpx.Timeout(120.0),
            limits=httpx.Limits(
                max_connections=max(16, concurrency),
            ),
        ) as client:

            tasks = [
                limited_request(client, item)
                for item in workload
            ]

            results = await __import__(
                "asyncio"
            ).gather(*tasks)

    finally:
        wall_seconds = time.perf_counter() - start
        gpu_stats = monitor.stop()

    request_df = pd.DataFrame(results)

    succeeded = request_df[
        request_df["success"] == True
    ].copy()

    completed = len(succeeded)
    failed = len(request_df) - completed

    # A failed request is not a successful throughput result.
    if failed:
        raise RuntimeError(
            f"{failed} requests failed for {model_name}, "
            f"concurrency={concurrency}. "
            f"Errors: {request_df.loc[~request_df['success'], 'error'].tolist()}"
        )

    total_output_tokens = int(
        succeeded["output_tokens"].sum()
    )

    # Remove missing values before calculating TTFT.
    ttft = succeeded["ttft_seconds"].dropna()
    latencies = succeeded["latency_seconds"].dropna()

    summary = {
        "model": model_name,
        "concurrency": concurrency,
        "requests": completed,
        "wall_seconds": wall_seconds,
        "requests_per_second": (
            completed / wall_seconds
        ),
        "output_tokens": total_output_tokens,
        "output_tokens_per_second": (
            total_output_tokens / wall_seconds
        ),
        "ttft_mean_ms": (
            float(ttft.mean() * 1000)
            if len(ttft) else None
        ),
        "ttft_p50_ms": (
            float(ttft.quantile(0.50) * 1000)
            if len(ttft) else None
        ),
        "ttft_p95_ms": (
            float(ttft.quantile(0.95) * 1000)
            if len(ttft) else None
        ),
        "latency_mean_ms": float(
            latencies.mean() * 1000
        ),
        "latency_p95_ms": float(
            latencies.quantile(0.95) * 1000
        ),
        "gpu_peak_gib": gpu_stats["gpu_peak_gib"],
        "gpu_average_gib": gpu_stats["gpu_average_gib"],
        "failed_requests": failed,
    }

    request_df["model"] = model_name
    request_df["concurrency"] = concurrency

    return summary, request_df


In [19]:

# ============================================================
# Cell 15: Benchmark the GPTQ INT4 model
# ============================================================

import threading
import subprocess
import numpy as np
import pandas as pd
import httpx

from tqdm.auto import tqdm


# ============================================================
# 1. Restore GPU memory monitoring if unavailable
# ============================================================

if "GPUMemoryMonitor" not in globals():

    class GPUMemoryMonitor:
        """
        Monitor NVIDIA GPU memory usage during inference.

        Uses nvidia-smi to measure total GPU memory usage.

        Parameters
        ----------
        interval_seconds : float
            Time between GPU memory measurements.

        Attributes
        ----------
        samples : list[float]
            Recorded memory values in MiB.
        """

        def __init__(self, interval_seconds=0.5):
            self.interval_seconds = interval_seconds
            self.samples = []
            self._stop = threading.Event()
            self._thread = None

        def _sample(self):
            """Collect GPU memory measurements."""
            while not self._stop.is_set():
                try:
                    result = subprocess.run(
                        [
                            "nvidia-smi",
                            "--query-gpu=memory.used",
                            "--format=csv,noheader,nounits",
                        ],
                        capture_output=True,
                        text=True,
                        check=True,
                        timeout=5,
                    )

                    memory_mib = float(
                        result.stdout.strip().splitlines()[0]
                    )

                    self.samples.append(memory_mib)

                except (
                    OSError,
                    ValueError,
                    subprocess.SubprocessError,
                ):
                    pass

                self._stop.wait(self.interval_seconds)

        def start(self):
            """Start GPU memory monitoring."""
            self._stop.clear()

            self._thread = threading.Thread(
                target=self._sample,
                daemon=True,
            )

            self._thread.start()

        def stop(self):
            """
            Stop monitoring and summarize GPU memory.

            Returns
            -------
            dict
                Peak and average GPU memory in GiB.
            """
            self._stop.set()

            if self._thread is not None:
                self._thread.join(timeout=10)

            if not self.samples:
                return {
                    "gpu_peak_gib": None,
                    "gpu_average_gib": None,
                }

            memory_gib = np.asarray(self.samples) / 1024

            return {
                "gpu_peak_gib": float(memory_gib.max()),
                "gpu_average_gib": float(memory_gib.mean()),
            }


# ============================================================
# 2. Verify benchmark dependencies
# ============================================================

required_components = [
    "run_concurrency_benchmark",
    "stream_completion",
    "benchmark_workloads",
    "CONCURRENCY_LEVELS",
    "RESULTS_DIR",
    "make_prompt",
]

missing = [
    name for name in required_components
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing notebook components: "
        + ", ".join(missing)
        + ". Run their original definition cells first."
    )

assert active_model_name == "gptq_int4", (
    "GPTQ INT4 server is not marked as active."
)

assert server_is_healthy(), (
    "vLLM server is not responding."
)

print("GPU memory monitor ready.")
print("All benchmark dependencies available.")
print("GPTQ inference server healthy.")


# ============================================================
# 3. Warm up the vLLM server
# ============================================================

async def warmup_server():
    """
    Warm up vLLM before benchmarking.

    Returns
    -------
    None
        Performs one non-benchmark inference request.
    """
    warmup_item = {
        "id": "warmup",
        "task": "medqa",
        "prompt": make_prompt({
            "task": "medqa",
            "prompt": (
                "Which vitamin deficiency causes scurvy?\n"
                "A. Vitamin A\n"
                "B. Vitamin B12\n"
                "C. Vitamin C\n"
                "D. Vitamin D"
            ),
        }),
    }

    async with httpx.AsyncClient(
        timeout=httpx.Timeout(120.0)
    ) as client:

        await stream_completion(
            client,
            warmup_item,
        )


print("\nWarming up GPTQ model...")

await warmup_server()

print("Warm-up completed.")


# ============================================================
# 4. Benchmark concurrency levels
# ============================================================

benchmark_summaries = []
benchmark_request_records = []

for concurrency in CONCURRENCY_LEVELS:

    print(
        f"\nBenchmarking concurrency = {concurrency}"
    )

    summary, requests = await run_concurrency_benchmark(
        model_name="gptq_int4",
        concurrency=concurrency,
        workload=benchmark_workloads[concurrency],
    )

    benchmark_summaries.append(summary)
    benchmark_request_records.append(requests)

    print(
        f"Concurrency {concurrency}: "
        f"{summary['output_tokens_per_second']:.2f} "
        f"output tok/s | "
        f"TTFT p50: {summary['ttft_p50_ms']:.2f} ms"
    )

    # Save progress after each completed workload.
    pd.DataFrame(benchmark_summaries).to_csv(
        RESULTS_DIR / "benchmark_partial_results.csv",
        index=False,
    )


# ============================================================
# 5. Display final benchmark results
# ============================================================

gptq_benchmark_df = pd.DataFrame(
    benchmark_summaries
)

print("\n" + "=" * 60)
print("GPTQ INT4 — vLLM BENCHMARK RESULTS")
print("=" * 60)

display(gptq_benchmark_df)

print("\nGPTQ benchmarking completed successfully.")


GPU memory monitor ready.
All benchmark dependencies available.
GPTQ inference server healthy.

Warming up GPTQ model...
Warm-up completed.

Benchmarking concurrency = 1
Concurrency 1: 38.60 output tok/s | TTFT p50: 47.17 ms

Benchmarking concurrency = 4
Concurrency 4: 71.24 output tok/s | TTFT p50: 96.38 ms

Benchmarking concurrency = 8
Concurrency 8: 85.01 output tok/s | TTFT p50: 131.55 ms

GPTQ INT4 — vLLM BENCHMARK RESULTS


,model,concurrency,requests,wall_seconds,requests_per_second,output_tokens,output_tokens_per_second,ttft_mean_ms,ttft_p50_ms,ttft_p95_ms,latency_mean_ms,latency_p95_ms,gpu_peak_gib,gpu_average_gib,failed_requests
0,gptq_int4,1,16,0.829110,19.297796,32,38.595592,46.717977,47.169899,53.059615,50.019349,56.297714,63.240234,63.224609,0
1,gptq_int4,4,16,0.449166,35.621573,32,71.243146,96.843504,96.380381,108.160838,104.084001,112.519575,63.240234,63.240234,0
2,gptq_int4,8,16,0.376424,42.505310,32,85.010621,138.634811,131.550186,177.243001,170.092413,213.835281,63.367188,63.367188,0



GPTQ benchmarking completed successfully.


In [20]:

# ============================================================
# Cell 16: Benchmark merged BF16 reference model
# ============================================================

if "merged_bf16" in STAGED_MODELS:

    # Release GPTQ GPU resources before loading BF16.
    stop_vllm_server()

    start_vllm_server("merged_bf16")

    await warmup_server()

    for concurrency in CONCURRENCY_LEVELS:
        summary, requests = await run_concurrency_benchmark(
            model_name="merged_bf16",
            concurrency=concurrency,
            workload=benchmark_workloads[concurrency],
        )

        benchmark_summaries.append(summary)
        benchmark_request_records.append(requests)

        print(
            f"BF16 concurrency {concurrency}: "
            f"{summary['output_tokens_per_second']:.2f} "
            f"output tok/s"
        )

        pd.DataFrame(benchmark_summaries).to_csv(
            RESULTS_DIR / "benchmark_partial_results.csv",
            index=False,
        )

else:
    print(
        "Merged BF16 checkpoint not available. "
        "GPTQ benchmark is complete, but the "
        "BF16 performance comparison is pending."
    )

print("Benchmark runs completed.")


Merged BF16 checkpoint not available. GPTQ benchmark is complete, but the BF16 performance comparison is pending.
Benchmark runs completed.


In [21]:

# ============================================================
# Cell 17: Compare BF16 and GPTQ inference performance
# ============================================================

benchmark_df = pd.DataFrame(
    benchmark_summaries
)

request_df = pd.concat(
    benchmark_request_records,
    ignore_index=True,
)

benchmark_df.to_csv(
    RESULTS_DIR / "vllm_benchmark_summary.csv",
    index=False,
)

request_df.to_csv(
    RESULTS_DIR / "vllm_request_metrics.csv",
    index=False,
)

required_models = {
    "gptq_int4",
    "merged_bf16",
}

has_comparison = required_models.issubset(
    set(benchmark_df["model"])
)

if has_comparison:
    comparison_rows = []

    for concurrency in CONCURRENCY_LEVELS:
        subset = benchmark_df[
            benchmark_df["concurrency"] == concurrency
        ]

        bf16 = subset[
            subset["model"] == "merged_bf16"
        ].iloc[0]

        gptq = subset[
            subset["model"] == "gptq_int4"
        ].iloc[0]

        throughput_speedup = (
            gptq["output_tokens_per_second"]
            / bf16["output_tokens_per_second"]
        )

        ttft_ratio = (
            bf16["ttft_p50_ms"]
            / gptq["ttft_p50_ms"]
        )

        comparison_rows.append({
            "concurrency": concurrency,
            "bf16_output_tps": (
                bf16["output_tokens_per_second"]
            ),
            "gptq_output_tps": (
                gptq["output_tokens_per_second"]
            ),
            "throughput_speedup_x": (
                throughput_speedup
            ),
            "bf16_ttft_p50_ms": bf16["ttft_p50_ms"],
            "gptq_ttft_p50_ms": gptq["ttft_p50_ms"],
            "ttft_speedup_x": ttft_ratio,
            "bf16_peak_gpu_gib": bf16["gpu_peak_gib"],
            "gptq_peak_gpu_gib": gptq["gpu_peak_gib"],
        })

    speedup_df = pd.DataFrame(comparison_rows)

    speedup_df.to_csv(
        RESULTS_DIR / "bf16_vs_gptq_speedups.csv",
        index=False,
    )

    display(speedup_df)

else:
    speedup_df = None

    print(
        "No BF16 speedup comparison available. "
        "Preserving measured GPTQ metrics."
    )

display(benchmark_df)


No BF16 speedup comparison available. Preserving measured GPTQ metrics.


,model,concurrency,requests,wall_seconds,requests_per_second,output_tokens,output_tokens_per_second,ttft_mean_ms,ttft_p50_ms,ttft_p95_ms,latency_mean_ms,latency_p95_ms,gpu_peak_gib,gpu_average_gib,failed_requests
0,gptq_int4,1,16,0.829110,19.297796,32,38.595592,46.717977,47.169899,53.059615,50.019349,56.297714,63.240234,63.224609,0
1,gptq_int4,4,16,0.449166,35.621573,32,71.243146,96.843504,96.380381,108.160838,104.084001,112.519575,63.240234,63.240234,0
2,gptq_int4,8,16,0.376424,42.505310,32,85.010621,138.634811,131.550186,177.243001,170.092413,213.835281,63.367188,63.367188,0


In [22]:

# ============================================================
# Cell 18: Prefix caching comparison
# ============================================================

RUN_PREFIX_CACHE_EXPERIMENT = True

prefix_cache_results = []

if RUN_PREFIX_CACHE_EXPERIMENT:

    # The cached result was already measured in Cell 15.
    cached_result = benchmark_df[
        (benchmark_df["model"] == "gptq_int4")
        & (benchmark_df["concurrency"] == 4)
    ].iloc[0].to_dict()

    cached_result["prefix_caching"] = True

    prefix_cache_results.append(cached_result)

    # Shut down the current model and disable caching.
    stop_vllm_server()

    ENABLE_PREFIX_CACHING = False

    start_vllm_server("gptq_int4")

    await warmup_server()

    uncached_summary, uncached_requests = (
        await run_concurrency_benchmark(
            model_name="gptq_int4",
            concurrency=4,
            workload=benchmark_workloads[4],
        )
    )

    uncached_summary["prefix_caching"] = False

    prefix_cache_results.append(uncached_summary)

    # Restore the default for any future server runs.
    stop_vllm_server()
    ENABLE_PREFIX_CACHING = True

    prefix_cache_df = pd.DataFrame(
        prefix_cache_results
    )

    display(
        prefix_cache_df[
            [
                "prefix_caching",
                "ttft_p50_ms",
                "ttft_p95_ms",
                "output_tokens_per_second",
                "gpu_peak_gib",
            ]
        ]
    )

    prefix_cache_df.to_csv(
        RESULTS_DIR / "prefix_caching_comparison.csv",
        index=False,
    )

else:
    prefix_cache_df = None
    print("Prefix caching experiment skipped.")


Starting model: gptq_int4
vLLM server is ready.


,prefix_caching,ttft_p50_ms,ttft_p95_ms,output_tokens_per_second,gpu_peak_gib
0,True,96.380381,108.160838,71.243146,63.240234
1,False,104.247069,113.238770,67.029753,63.806641


In [ ]:

# ============================================================
# Cell 19: Visualize throughput and latency
# ============================================================

def plot_benchmark_metric(
    results,
    column,
    ylabel,
    title,
    filename,
):
    """
    Plot a benchmark metric against request concurrency.

    Parameters
    ----------
    results : pandas.DataFrame
        Aggregated model benchmark results.
    column : str
        Metric to visualize.
    ylabel : str
        Y-axis label.
    title : str
        Figure title.
    filename : str
        Output image filename.

    Returns
    -------
    None
        Displays and saves the figure.
    """
    fig, ax = plt.subplots(figsize=(9, 5))

    for model_name, group in results.groupby("model"):
        group = group.sort_values("concurrency")

        ax.plot(
            group["concurrency"],
            group[column],
            marker="o",
            linewidth=2,
            label=model_name,
        )

    ax.set_xlabel("Concurrent requests")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(CONCURRENCY_LEVELS)
    ax.grid(alpha=0.25)
    ax.legend()

    plt.tight_layout()

    plt.savefig(
        RESULTS_DIR / filename,
        dpi=150,
    )

    plt.show()


plot_benchmark_metric(
    benchmark_df,
    column="output_tokens_per_second",
    ylabel="Generated tokens / second",
    title="Clinical QA — vLLM Output Throughput",
    filename="vllm_throughput.png",
)

plot_benchmark_metric(
    benchmark_df,
    column="ttft_p50_ms",
    ylabel="Median TTFT (ms)",
    title="Clinical QA — Time to First Token",
    filename="vllm_ttft.png",
)

plot_benchmark_metric(
    benchmark_df,
    column="latency_mean_ms",
    ylabel="Mean response latency (ms)",
    title="Clinical QA — End-to-End Latency",
    filename="vllm_latency.png",
)


In [ ]:

# ============================================================
# Cell 20: Export complete inference benchmark report
# ============================================================

def make_json_safe(value):
    """
    Convert NumPy and pandas values to JSON-safe types.

    Parameters
    ----------
    value : object
        Arbitrary benchmark value.

    Returns
    -------
    object
        Value compatible with standard JSON serialization.
    """
    if isinstance(value, dict):
        return {
            str(key): make_json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [make_json_safe(item) for item in value]

    if isinstance(value, np.generic):
        return value.item()

    if pd.isna(value):
        return None

    return value


report = {
    "experiment": "clinical-qa-vllm-inference",
    "base_model": "Qwen/Qwen3-4B",
    "fine_tuning_method": "QLoRA",
    "quantization_method": "GPTQ INT4",
    "quantization_group_size": 128,
    "gpu": GPU_NAME,
    "gpu_total_gib": GPU_MEMORY_GB,
    "pytorch_version": torch.__version__,
    "vllm_version": vllm.__version__,
    "benchmark_configuration": {
        "concurrency_levels": CONCURRENCY_LEVELS,
        "requests_per_level": QUESTIONS_PER_LEVEL,
        "max_generated_tokens": MAX_NEW_TOKENS,
        "max_model_length": MAX_MODEL_LEN,
        "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
        "max_num_seqs": MAX_NUM_SEQS,
        "decoding": "greedy",
        "thinking_enabled": False,
    },
    "previous_validation_results": {
        "qlora_macro_accuracy": 0.75,
        "gptq_macro_accuracy": 0.725,
        "note": (
            "These are Notebook 03/04 validation results, "
            "not new accuracy measurements."
        ),
    },
    "checkpoint_sizes_gib": checkpoint_sizes,
    "benchmark_results": benchmark_df.to_dict(
        orient="records"
    ),
    "bf16_vs_gptq": (
        speedup_df.to_dict(orient="records")
        if speedup_df is not None
        else None
    ),
    "prefix_cache_results": (
        prefix_cache_df.to_dict(orient="records")
        if prefix_cache_df is not None
        else None
    ),
}

report = make_json_safe(report)

REPORT_PATH = (
    RESULTS_DIR / "vllm_inference_report.json"
)

with open(
    REPORT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        report,
        file,
        indent=4,
        allow_nan=False,
    )

print("Benchmark report saved:", REPORT_PATH)
print("Measured model configurations:")
print(benchmark_df["model"].unique().tolist())
print("Total benchmark runs:", len(benchmark_df))


In [25]:

# ============================================================
# Part 06 — Cell 1: Verify existing server and dependencies
# ============================================================

import importlib.util
import subprocess
import sys

import httpx

VLLM_BASE_URL = "http://127.0.0.1:8000/v1"
MODEL_NAME = "clinical-qa"

# Check the server that was started in Notebook 05.
response = httpx.get(
    f"{VLLM_BASE_URL}/models",
    timeout=10
)

response.raise_for_status()

available_models = [
    item["id"]
    for item in response.json()["data"]
]

assert MODEL_NAME in available_models, (
    f"Expected {MODEL_NAME}, found {available_models}"
)

print("vLLM server: READY")
print("Available models:", available_models)

# Install only missing application packages.
required_packages = {
    "fastapi": "fastapi",
    "gradio": "gradio",
    "uvicorn": "uvicorn",
}

missing = [
    package
    for module, package in required_packages.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Missing packages:", missing)
    print(
        "Install them using %pip install, then restart "
        "the session only if required."
    )
else:
    print("FastAPI, Gradio and Uvicorn: READY")


vLLM server: READY
Available models: ['clinical-qa']
FastAPI, Gradio and Uvicorn: READY


In [26]:

# ============================================================
# Part 06 — Cell 2: Create the deployment application
# ============================================================

from pathlib import Path

APP_FILE = Path("/content/clinical_qa_app.py")

app_source = r'''
"""
Clinical Q&A Research Application.

FastAPI provides a prediction endpoint.
Gradio provides an interactive web interface.
vLLM handles GPU inference using our GPTQ model.

Research and educational use only.
"""

import os
import re
import time

import gradio as gr
import httpx

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field


# ============================================================
# Configuration
# ============================================================

VLLM_BASE_URL = os.getenv(
    "VLLM_BASE_URL",
    "http://127.0.0.1:8000/v1"
).rstrip("/")

MODEL_NAME = os.getenv(
    "VLLM_MODEL_NAME",
    "clinical-qa"
)

MAX_NEW_TOKENS = 24


# ============================================================
# Task instructions — same as Notebook 03
# ============================================================

SYSTEM_PROMPTS = {
    "medqa": (
        "You are answering educational medical "
        "multiple-choice questions. Select the "
        "single best answer. Begin with A, B, C, or D."
    ),
    "pubmedqa": (
        "You answer biomedical research questions "
        "using the provided abstract evidence. "
        "Begin your answer with yes, no, or maybe."
    ),
}


# ============================================================
# Pydantic request validation
# ============================================================

class QuestionRequest(BaseModel):
    """Validate one clinical question."""

    task: str = Field(
        ...,
        description="medqa or pubmedqa"
    )

    question: str = Field(
        ...,
        min_length=5,
        max_length=12000
    )


# ============================================================
# Answer extraction
# ============================================================

def extract_answer(text, task):
    """
    Extract a valid answer from the generated response.

    Parameters
    ----------
    text : str
        Raw model response.
    task : str
        MedQA or PubMedQA.

    Returns
    -------
    str or None
        Normalized predicted label.
    """
    if not isinstance(text, str):
        return None

    text = re.sub(
        r"<think>.*?</think>",
        "",
        text,
        flags=re.DOTALL | re.IGNORECASE
    ).strip()

    if "<think>" in text.lower():
        return None

    if task == "medqa":
        patterns = [
            r"^\s*\(?([ABCD])\)?(?:[.):]|\s|$)",
            r"^\s*(?:final\s+answer|correct\s+answer|answer)"
            r"\s*(?:is|:|-)?\s*\(?([ABCD])\)?\b",
        ]

        for pattern in patterns:
            match = re.search(
                pattern,
                text,
                flags=re.IGNORECASE
            )
            if match:
                return match.group(1).upper()

    elif task == "pubmedqa":
        match = re.match(
            r"^\s*(?:answer\s*:\s*)?"
            r"(yes|no|maybe)\b",
            text,
            flags=re.IGNORECASE
        )

        if match:
            return match.group(1).lower()

    return None


# ============================================================
# vLLM inference
# ============================================================

async def run_inference(task, question):
    """
    Send a clinical QA request to the existing vLLM server.

    Parameters
    ----------
    task : str
        'medqa' or 'pubmedqa'.
    question : str
        Question with options or abstract evidence.

    Returns
    -------
    dict
        Model prediction, raw response and latency.

    Raises
    ------
    ValueError
        When an invalid task is supplied.
    RuntimeError
        When vLLM inference fails.
    """
    if task not in SYSTEM_PROMPTS:
        raise ValueError(f"Unsupported task: {task}")

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPTS[task]
            },
            {
                "role": "user",
                "content": question
            }
        ],
        "temperature": 0,
        "max_tokens": MAX_NEW_TOKENS,
        "stream": False,
        "chat_template_kwargs": {
            "enable_thinking": False
        }
    }

    start_time = time.perf_counter()

    try:
        async with httpx.AsyncClient(
            timeout=120
        ) as client:

            response = await client.post(
                f"{VLLM_BASE_URL}/chat/completions",
                json=payload
            )

            response.raise_for_status()
            result = response.json()

    except httpx.HTTPError as exc:
        raise RuntimeError(
            "vLLM inference request failed."
        ) from exc

    latency_ms = (
        time.perf_counter() - start_time
    ) * 1000

    raw_response = (
        result["choices"][0]["message"]["content"] or ""
    ).strip()

    prediction = extract_answer(
        raw_response,
        task
    )

    return {
        "task": task,
        "prediction": prediction,
        "raw_response": raw_response,
        "latency_ms": round(latency_ms, 2),
        "valid_answer": prediction is not None,
        "model": MODEL_NAME
    }


# ============================================================
# FastAPI application
# ============================================================

app = FastAPI(
    title="Clinical Q&A API",
    version="1.0.0",
    description="Qwen3-4B GPTQ INT4 served through vLLM"
)


@app.get("/health")
async def health():
    """Check application and vLLM availability."""
    try:
        async with httpx.AsyncClient(
            timeout=5
        ) as client:
            response = await client.get(
                f"{VLLM_BASE_URL}/models"
            )

        available = response.status_code == 200

    except httpx.HTTPError:
        available = False

    return {
        "application": "running",
        "vllm_connected": available,
        "model": MODEL_NAME
    }


@app.post("/api/predict")
async def predict(request: QuestionRequest):
    """Return a structured clinical QA prediction."""
    if request.task not in SYSTEM_PROMPTS:
        raise HTTPException(
            status_code=400,
            detail="Task must be medqa or pubmedqa."
        )

    try:
        return await run_inference(
            request.task,
            request.question
        )

    except RuntimeError:
        raise HTTPException(
            status_code=503,
            detail="Inference server is unavailable."
        )


# ============================================================
# Gradio interface
# ============================================================

async def submit_question(task, question):
    """
    Generate an answer for the Gradio interface.

    Returns
    -------
    tuple
        Parsed prediction, raw response, latency, status.
    """
    if not question or not question.strip():
        return "", "", "", "Please enter a question."

    try:
        result = await run_inference(
            task,
            question.strip()
        )

        return (
            result["prediction"] or "INVALID",
            result["raw_response"],
            f'{result["latency_ms"]:.2f} ms',
            "Completed" if result["valid_answer"]
            else "Unable to parse answer"
        )

    except (RuntimeError, ValueError):
        return (
            "Unavailable",
            "",
            "",
            "Inference failed. Check the vLLM server."
        )


with gr.Blocks(
    title="Clinical Q&A Assistant"
) as demo:

    gr.Markdown(
        """
        # Clinical Q&A Assistant

        **Qwen3-4B | QLoRA | GPTQ INT4 | vLLM**

        Educational medical questions and biomedical
        literature question answering.

        **Research only — not for medical decisions.**
        Do not submit patient-identifiable information.
        """
    )

    task_input = gr.Radio(
        choices=[
            ("MedQA — Multiple Choice", "medqa"),
            ("PubMedQA — Research Question", "pubmedqa")
        ],
        value="medqa",
        label="Question type"
    )

    question_input = gr.Textbox(
        label="Clinical question",
        lines=7,
        placeholder=(
            "Enter a question with answer options "
            "or an abstract with a research question."
        )
    )

    submit_button = gr.Button(
        "Generate Answer",
        variant="primary"
    )

    prediction_output = gr.Textbox(
        label="Predicted answer"
    )

    raw_output = gr.Textbox(
        label="Raw model output"
    )

    latency_output = gr.Textbox(
        label="Inference latency"
    )

    status_output = gr.Textbox(
        label="Status"
    )

    submit_button.click(
        fn=submit_question,
        inputs=[task_input, question_input],
        outputs=[
            prediction_output,
            raw_output,
            latency_output,
            status_output
        ]
    )

    gr.Examples(
        examples=[
            [
                "medqa",
                "Which vitamin deficiency causes scurvy?\n"
                "A. Vitamin A\n"
                "B. Vitamin B12\n"
                "C. Vitamin C\n"
                "D. Vitamin D"
            ],
            [
                "pubmedqa",
                "Question: Can tailored interventions increase "
                "mammography use among HMO women?\n"
                "Context: A study reported an increase in "
                "mammography use following tailored outreach."
            ]
        ],
        inputs=[task_input, question_input]
    )


# ============================================================
# Mount Gradio in FastAPI
# ============================================================

app = gr.mount_gradio_app(
    app,
    demo,
    path="/app"
)
'''

APP_FILE.write_text(app_source, encoding="utf-8")

print("Application created:", APP_FILE)
print("File size:", APP_FILE.stat().st_size, "bytes")


Application created: /content/clinical_qa_app.py
File size: 9827 bytes


In [27]:

# ============================================================
# Part 06 — Cell 3: Launch FastAPI + Gradio
# ============================================================

import os
import signal
import subprocess
import time

import httpx

APP_PORT = 7860
APP_URL = f"http://127.0.0.1:{APP_PORT}"

app_process = None
APP_LOG = Path("/content/clinical_qa_app.log")


def start_web_app():
    """
    Start FastAPI and Gradio in a separate process.

    Returns
    -------
    subprocess.Popen
        Running web application process.

    Raises
    ------
    RuntimeError
        If the web application fails during startup.
    """
    global app_process

    # Confirm the original vLLM server is still available.
    assert httpx.get(
        "http://127.0.0.1:8000/health",
        timeout=10
    ).status_code == 200, (
        "vLLM server is not available."
    )

    # Do not start another app on an occupied port.
    try:
        response = httpx.get(
            f"{APP_URL}/health",
            timeout=2
        )

        if response.status_code == 200:
            print("Web application is already running.")
            return app_process

    except httpx.HTTPError:
        pass

    command = [
        sys.executable,
        "-m",
        "uvicorn",
        "clinical_qa_app:app",
        "--host", "0.0.0.0",
        "--port", str(APP_PORT)
    ]

    with open(APP_LOG, "w") as log:
        app_process = subprocess.Popen(
            command,
            cwd="/content",
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True
        )

    # Check for successful application startup.
    for _ in range(30):
        if app_process.poll() is not None:
            raise RuntimeError(
                "Web application startup failed:\n"
                + APP_LOG.read_text(errors="replace")[-4000:]
            )

        try:
            response = httpx.get(
                f"{APP_URL}/health",
                timeout=3
            )

            if response.status_code == 200:
                print("FastAPI + Gradio: READY")
                print("Application:", APP_URL)
                return app_process

        except httpx.HTTPError:
            pass

        time.sleep(2)

    raise RuntimeError(
        "Application did not become ready.\n"
        + APP_LOG.read_text(errors="replace")[-4000:]
    )


start_web_app()


FastAPI + Gradio: READY
Application: http://127.0.0.1:7860


<Popen: returncode: None args: ['/usr/bin/python3', '-m', 'uvicorn', 'clinic...>

In [28]:

# ============================================================
# Part 06 — Cell 4: Test end-to-end clinical QA inference
# ============================================================

import httpx

test_question = {
    "task": "medqa",
    "question": (
        "Which vitamin deficiency causes scurvy?\n"
        "A. Vitamin A\n"
        "B. Vitamin B12\n"
        "C. Vitamin C\n"
        "D. Vitamin D"
    )
}

response = httpx.post(
    "http://127.0.0.1:7860/api/predict",
    json=test_question,
    timeout=120
)

response.raise_for_status()

result = response.json()

print("=" * 50)
print("CLINICAL QA — END-TO-END TEST")
print("=" * 50)
print("Model:", result["model"])
print("Prediction:", result["prediction"])
print("Raw response:", result["raw_response"])
print("Latency:", result["latency_ms"], "ms")
print("Valid answer:", result["valid_answer"])


CLINICAL QA — END-TO-END TEST
Model: clinical-qa
Prediction: C
Raw response: C
Latency: 87.29 ms
Valid answer: True


In [29]:

# ============================================================
# Part 06 — Cell 5: Open the Gradio interface
# ============================================================

from google.colab import output

# Open the running FastAPI + Gradio application
# through Colab's notebook port proxy.
output.serve_kernel_port_as_window(7860)


Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [30]:
import requests

for name, url in {
    "vLLM": "http://127.0.0.1:8000/health",
    "FastAPI": "http://127.0.0.1:7860/health",
    "Gradio": "http://127.0.0.1:7860/app",
}.items():
    try:
        response = requests.get(url, timeout=10)
        print(f"{name}: {response.status_code}")
    except Exception as e:
        print(f"{name}: ERROR - {e}")

vLLM: 200
FastAPI: 200
Gradio: 200


In [31]:

# ============================================================
# Fix: Generate a fresh Colab URL for Gradio
# ============================================================

from google.colab import output
import requests

# Confirm the application is running.
response = requests.get(
    "http://127.0.0.1:7860/app",
    timeout=10,
)

print("Local Gradio status:", response.status_code)

if response.status_code == 200:
    # Get a proxy URL for the CURRENT Colab session.
    proxy_url = output.eval_js(
        "google.colab.kernel.proxyPort(7860)"
    )

    gradio_url = proxy_url.rstrip("/") + "/app"

    print("\nOpen this NEW URL:")
    print(gradio_url)
else:
    print("The Gradio application is not responding locally.")


Local Gradio status: 200

Open this NEW URL:
https://7860-gpu-a100-hm-ass1c0-6hjlx9g7f8qo-c.asia-southeast1.prod.r.colab.dev/app


In [32]:

# ============================================================
# Display Gradio directly inside Google Colab
# ============================================================

from google.colab import output

# Display the existing FastAPI + Gradio application.
# Port 7860 = FastAPI
# /app = Mounted Gradio interface

output.serve_kernel_port_as_iframe(
    7860,
    path="/app/",
    height=800,
)


<IPython.core.display.Javascript object>

In [34]:

# ============================================================
# Diagnose Gradio routing through Colab
# ============================================================

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

base = "http://127.0.0.1:7860"
app_url = f"{base}/app/"

response = requests.get(app_url, timeout=15)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

# Check Gradio's static assets and configuration endpoint.
routes = {
    "Gradio homepage": "/app/",
    "Gradio config": "/app/config",
    "Gradio root config": "/config",
}

for name, route in routes.items():
    try:
        result = requests.get(base + route, timeout=10)
        print(f"{name}: HTTP {result.status_code}")
    except requests.RequestException as exc:
        print(f"{name}: ERROR - {exc}")

# Check external JavaScript and stylesheet paths.
for tag in soup.find_all(["script", "link"]):
    asset = tag.get("src") or tag.get("href")

    if not asset:
        continue

    asset_url = urljoin(app_url, asset)

    if not asset_url.startswith(base):
        continue

    try:
        result = requests.get(asset_url, timeout=10)
        print(
            f"Asset: HTTP {result.status_code} "
            f"{asset_url.replace(base, '')}"
        )
    except requests.RequestException as exc:
        print(f"Asset ERROR: {exc}")


Gradio homepage: HTTP 200
Gradio config: HTTP 200
Gradio root config: HTTP 404
Asset: HTTP 404 /manifest.json
Asset: HTTP 200 /app/assets/index-DphvyggB.js
Asset: HTTP 200 /app/assets/index-DiEEvXS6.css


In [35]:

# ============================================================
# Deployment Fix: Open existing Gradio app via share link
# ============================================================

import sys

# Import the existing application from Part 06.
sys.path.insert(0, "/content")

from clinical_qa_app import demo

# Reuse the running vLLM server on port 8000.
# This launches only the Gradio interface, not another model.
demo.launch(
    server_name="0.0.0.0",
    server_port=7861,
    share=True,
    inline=False,
    prevent_thread_lock=True,
    show_error=True
)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2928242ab48174c47c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [37]:
from huggingface_hub import login

login()

In [38]:
from pathlib import Path
from huggingface_hub import HfApi

MODEL_DIR = Path(
    "/content/drive/MyDrive/clinical-qa-finetuning-v2/"
    "models/qwen3-clinical-gptq-int4"
)

assert MODEL_DIR.exists(), "Model directory not found"
assert (MODEL_DIR / "config.json").exists()
assert list(MODEL_DIR.glob("*.safetensors"))

api = HfApi()

username = api.whoami()["name"]

repo_id = f"{username}/Qwen3-4B-Clinical-QA-GPTQ-INT4"

# Create the model repository.
api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    private=False,
    exist_ok=True,
)

# Upload all model files, including configuration and tokenizer.
api.upload_folder(
    folder_path=str(MODEL_DIR),
    repo_id=repo_id,
    repo_type="model",
    commit_message="Upload Qwen3-4B Clinical QA GPTQ INT4 model",
)

print("Model uploaded successfully!")
print(f"https://huggingface.co/{repo_id}")

Model uploaded successfully!
https://huggingface.co/PardhuSSK/Qwen3-4B-Clinical-QA-GPTQ-INT4


In [39]:

# ============================================================
# Model Card — Cell 1: Create README.md
# ============================================================

from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/clinical-qa-finetuning-v2"
)

MODEL_CARD_PATH = PROJECT_DIR / "README_model_card.md"

model_card = """---
language:
- en
base_model: Qwen/Qwen3-4B
pipeline_tag: text-generation
tags:
- qwen3
- medical-question-answering
- biomedical-nlp
- clinical-qa
- qlora
- gptq
- 4bit
- medqa
- pubmedqa
- vllm
---

# Qwen3-4B Clinical QA — GPTQ INT4

A domain-adapted, 4-bit quantized language model for
educational medical multiple-choice questions and
biomedical research question answering.

This model is based on **Qwen/Qwen3-4B**, fine-tuned
using QLoRA on a combination of MedQA and PubMedQA
training data, then merged and quantized using GPTQ.

The project explores the full machine learning lifecycle:

**Data Preparation → QLoRA Fine-Tuning → Evaluation
→ GPTQ Quantization → vLLM Inference → Deployment**

> **Medical disclaimer:** This model is intended for
> research and educational use only. It is not a
> clinically validated medical device and must not
> be used for diagnosis, treatment, or patient care.

## 1. Model Overview

| Property | Description |
|---|---|
| Base model | Qwen/Qwen3-4B |
| Model architecture | Qwen3ForCausalLM |
| Application | Clinical and biomedical question answering |
| Training technique | QLoRA |
| Quantization | GPTQ, 4-bit |
| Quantization group size | 128 |
| Language | English |
| Serving engine | vLLM |
| Published checkpoint | GPTQ INT4 |
| Checkpoint size | Approximately 2.497 GiB |

### Supported Tasks

**MedQA**

The model receives a medical multiple-choice question
and predicts a single answer choice:

`A`, `B`, `C`, or `D`.

**PubMedQA**

The model receives a biomedical research question with
supporting abstract context and predicts:

`yes`, `no`, or `maybe`.

The model is trained to provide short, structured labels,
not comprehensive clinical explanations.

## 2. Training Data

The supervised fine-tuning dataset combines examples
from two medical question-answering benchmarks:

- **MedQA:** Medical multiple-choice questions.
- **PubMedQA:** Biomedical research questions with
  abstract-based context and categorical answers.

### Dataset Preparation

The project prepared approximately 12,000 supervised
fine-tuning records.

The final training run used:

| Setting | Value |
|---|---|
| Training records | 11,999 |
| Validation records | 64 |
| Maximum sequence length | 1,024 |
| Training epochs | 1 |
| Maximum training steps | 1,500 |

A separate evaluation selection was maintained for
MedQA and PubMedQA model comparisons.

This repository does not redistribute the underlying
medical benchmark datasets.

Users should consult the original dataset licenses
and usage conditions.

## 3. Fine-Tuning Configuration

Fine-tuning was performed using QLoRA, a
parameter-efficient technique combining low-rank
adaptation with quantized model loading.

| Hyperparameter | Value |
|---|---|
| Base model | Qwen3-4B |
| Method | QLoRA |
| LoRA rank | 16 |
| LoRA alpha | 32 |
| Learning rate | 2e-4 |
| Maximum training steps | 1,500 |
| Maximum sequence length | 1,024 |
| Training GPU | NVIDIA A100 |
| Final training loss | 0.4462 |
| Final validation loss | 0.59846 |

The fine-tuned adapter was preserved separately
before the quantization stage.

## 4. GPTQ Quantization

After fine-tuning, the LoRA adapter was merged into
the base model.

The resulting checkpoint was quantized using GPTQ
to reduce storage requirements and enable
more memory-efficient inference.

| Configuration | Value |
|---|---|
| Quantization method | GPTQ |
| Quantization library | GPTQModel |
| Weight precision | INT4 |
| Group size | 128 |
| Calibration samples | 256 |
| Calibration sequence length | 512 |
| Quantized checkpoint size | 2.497 GiB |

The uploaded model contains the quantized
standalone checkpoint rather than just the LoRA adapter.

## 5. Evaluation Results

The base, QLoRA fine-tuned, and GPTQ INT4 models
were evaluated on the same selected questions.

Evaluation samples:

- MedQA: 100 questions
- PubMedQA: 100 questions

### Accuracy Comparison

| Dataset | Base Qwen3-4B | Fine-Tuned QLoRA | GPTQ INT4 |
|---|---:|---:|---:|
| MedQA | 58% | 67% | 62% |
| PubMedQA | 43% | 83% | 83% |
| Macro-average | 50.5% | 75.0% | 72.5% |

### Observations

Compared with the original base model, GPTQ INT4
achieved:

- **MedQA:** +4 percentage points
- **PubMedQA:** +40 percentage points
- **Macro-average:** +22 percentage points

Compared with the unquantized fine-tuned model,
GPTQ INT4 showed:

- **MedQA:** -5 percentage points
- **PubMedQA:** No observed difference
- **Macro-average:** -2.5 percentage points

These results are based on selected 100-question
samples per dataset, not the full benchmark test sets.

The quantized model did not satisfy the project's
initial maximum 2-percentage-point regression
criterion for MedQA or macro-average accuracy.

It is therefore published as a research and
deployment-optimization candidate, not as a
quantization configuration proven to preserve
the fine-tuned model's full accuracy.

## 6. vLLM Inference Benchmark

The quantized checkpoint was served using vLLM
on an NVIDIA A100-SXM4-80GB GPU.

### Serving Configuration

| Parameter | Value |
|---|---|
| Serving engine | vLLM 0.33.0 |
| Maximum model length | 2,048 |
| Maximum concurrent sequences | 16 |
| GPU memory utilization target | 0.80 |
| Prefix caching | Enabled |
| Maximum generated tokens | 24 |

### Throughput and Latency

| Concurrent requests | Output tokens/sec | Median TTFT |
|---|---:|---:|
| 1 | 39.66 | 45.73 ms |
| 4 | 71.18 | 98.12 ms |
| 8 | 85.16 | 146.29 ms |

Each concurrency setting used 16 selected
clinical question-answering requests.

TTFT measures client-observed time to the first
non-empty generated text chunk.

The experiment did not establish a BF16-versus-GPTQ
inference speedup because a BF16 reference model
was not available during the benchmark.

These results describe the tested Colab GPU
environment and are not guaranteed for other hardware.

## 7. Running the Model with vLLM

Install vLLM in a compatible CUDA environment:

```bash
pip install vllm
```

Launch the published checkpoint:

```bash
vllm serve PardhuSSK/Qwen3-4B-Clinical-QA-GPTQ-INT4 \\
    --served-model-name clinical-qa \\
    --max-model-len 2048 \\
    --gpu-memory-utilization 0.80
```

The model exposes an OpenAI-compatible inference API.

Example Python request:

```python
import requests

messages = [
    {
        "role": "system",
        "content": (
            "You are answering educational medical "
            "multiple-choice questions. Select the "
            "single best answer. Begin with A, B, C, or D."
        ),
    },
    {
        "role": "user",
        "content": (
            "Which vitamin deficiency causes scurvy?\\n"
            "A. Vitamin A\\n"
            "B. Vitamin B12\\n"
            "C. Vitamin C\\n"
            "D. Vitamin D"
        ),
    },
]

response = requests.post(
    "http://127.0.0.1:8000/v1/chat/completions",
    json={
        "model": "clinical-qa",
        "messages": messages,
        "temperature": 0,
        "max_tokens": 24,
        "chat_template_kwargs": {
            "enable_thinking": False
        },
    },
    timeout=120,
)

response.raise_for_status()

print(response.json()["choices"][0]["message"]["content"])
```

The sample demonstrates the inference interface;
it is not a guarantee of a particular prediction.

## 8. Intended Use

This model is suitable for:

- Research on medical-domain language models.
- Educational question-answering experiments.
- Medical NLP benchmarking.
- Studying parameter-efficient fine-tuning.
- GPTQ quantization and inference optimization.
- Prototyping Gradio and FastAPI applications.

### Out-of-Scope Uses

Do not use this model for:

- Clinical diagnosis or treatment recommendations.
- Patient-specific medical decision-making.
- Emergency or triage decisions.
- Automated clinical workflow decisions.
- Processing confidential patient records without
  appropriate privacy safeguards.

## 9. Limitations

- The model can generate incorrect medical answers.
- Accuracy varies between MedQA and PubMedQA.
- GPTQ quantization reduced measured MedQA accuracy
  relative to the fine-tuned checkpoint.
- Evaluation used limited selected question samples.
- Performance on real patient cases was not evaluated.
- The model does not verify answers against live
  medical literature.
- The system does not provide calibrated clinical
  confidence probabilities.
- Biomedical information may become outdated.

Any real-world clinical use would require substantially
more validation, safety evaluation, governance,
and appropriate regulatory review.

## 10. Project Resources

**Base Model:**
https://huggingface.co/Qwen/Qwen3-4B

**Published GPTQ Model:**
https://huggingface.co/PardhuSSK/Qwen3-4B-Clinical-QA-GPTQ-INT4

**Source Code:**
GitHub repository link to be added after publication.

## 11. Acknowledgments

This project builds on Qwen3, Hugging Face Transformers,
PEFT/QLoRA, GPTQModel, and vLLM.

MedQA and PubMedQA are used as research benchmarks.

Users should review the original Qwen3 model license,
dataset licenses, and library licenses before reuse.

---

**Research project | Clinical QA | QLoRA | GPTQ | vLLM**
"""

MODEL_CARD_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_CARD_PATH.write_text(
    model_card,
    encoding="utf-8"
)

print("Model card created successfully!")
print("Location:", MODEL_CARD_PATH)
print("Characters:", len(model_card))


Model card created successfully!
Location: /content/drive/MyDrive/clinical-qa-finetuning-v2/README_model_card.md
Characters: 9130


In [40]:

# ============================================================
# Model Card — Cell 2: Upload README to Hugging Face
# ============================================================

from huggingface_hub import HfApi

api = HfApi()

REPO_ID = "PardhuSSK/Qwen3-4B-Clinical-QA-GPTQ-INT4"

# Confirm the locally generated README exists.
assert MODEL_CARD_PATH.is_file(), (
    f"Model card not found: {MODEL_CARD_PATH}"
)

# Upload only the model card.
commit = api.upload_file(
    path_or_fileobj=str(MODEL_CARD_PATH),
    path_in_repo="README.md",
    repo_id=REPO_ID,
    repo_type="model",
    commit_message=(
        "Add clinical QA model card with "
        "training, evaluation, and quantization results"
    ),
)

print("Model card uploaded successfully!")
print("Repository:")
print(f"https://huggingface.co/{REPO_ID}")
print("Commit:", commit.oid)


Model card uploaded successfully!
Repository:
https://huggingface.co/PardhuSSK/Qwen3-4B-Clinical-QA-GPTQ-INT4
Commit: 38ec45f0b2db636d2de8bd92af3954ee66467c9a
